# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [4]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds, repeating 8-12 times.\n\nThese exercises can help alleviate discomfort and prevent future episodes of lower back pain.'

In [11]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. Adequate sleep, typically 7-9 hours for adults, is crucial for physical repair, mental well-being, and cognitive functions such as memory and learning. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Poor sleep or conditions like insomnia can impair these processes, leading to health issues. Maintaining good sleep hygiene—such as a consistent sleep schedule, a relaxing bedtime routine, and creating an optimal sleep environment—can improve sleep quality and support overall health.'

In [12]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water and staying hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using essential oils like peppermint or lavender\n- Engaging in deep breathing exercises or progressive muscle relaxation\n- Taking short walks, especially in nature\n- Listening to calming music\n- Practicing mindfulness and meditation regularly\n\nThese strategies can help alleviate stress and reduce headache symptoms naturally.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees. Alternate between arching your back up (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n- **Bird Dog:** From your hands and knees, extend your opposite arm and leg while keeping your core engaged. Hold for about 5 seconds, then switch sides. Do 10 repetitions per side.\n- **Pelvic Tilts:** Lie on your back with knees bent. Flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and may prevent future issues.'

In [16]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a crucial role in overall health. Maintaining a consistent sleep schedule, creating a sleep-friendly environment, and practicing good sleep hygiene—such as a relaxing bedtime routine, limiting screen time before bed, and ensuring a comfortable, quiet, and dark bedroom—all contribute to better sleep quality. Good sleep supports immune function, mental health, and nutrient absorption, and helps regulate mood and energy levels. Therefore, quality sleep is foundational to overall wellness and can positively impact various aspects of health.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include practicing relaxation techniques such as deep breathing exercises and meditation. For headaches, staying well-hydrated, managing stress, ensuring adequate sleep, and avoiding known triggers like certain foods can help. Herbal teas like chamomile or valerian root may also promote relaxation and help reduce headache occurrence.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:
When usecase is about retrieveing based on exact search (e.g. timestamp, suppliment name, lab results) BM25 outperforms embedding based search. Whereas if questions are conceptual/causal - embedding performs better. 

Below I have used an exmaple question where in healthdoc it says 25-35 gms of fibre intake is recommended. With BM25, I could extract the relevant information whereas embedding based search failed.

In [18]:
bm25_retrieval_chain.invoke({"question" :"What is 25 grams"})["response"].content

'In the context provided, 25 grams refers to the recommended daily intake of fiber-rich foods to support digestive health. Specifically, it suggests eating fiber-rich foods totaling 25 grams daily.'

In [19]:
naive_retrieval_chain.invoke({"question" : "What is 25 grams"})["response"].content

'Twenty-five grams typically refers to a measurement of weight equal to 25 grams. It is roughly equivalent to about 0.88 ounces. In practical terms, it could be comparable to a small handful of some foods, like about a handful of nuts or a few slices of cheese.'

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [20]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [21]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees. Arch your back upwards (cat) and then let it sag downwards (cow). Repeat this movement 10-15 times.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for about 5 seconds, then switch sides. Perform 10 repetitions on each side.\n- **Pelvic Tilts:** Lie on your back with knees bent. Flatten your lower back against the floor by tightening your abdominal muscles and tilting your pelvis slightly upward. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and may prevent future episodes.'

In [23]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. According to the information provided, sleep is crucial for physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep, typically 7-9 hours for adults, is essential for these bodily processes to function properly. Additionally, maintaining an optimal sleep environment—such as keeping the room at a comfortable temperature, ensuring darkness and quietness, and having supportive bedding—can promote better sleep quality. Poor sleep or conditions like insomnia can negatively affect health, highlighting the importance of good sleep habits for overall well-being.'

In [24]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing exercises, engaging in progressive muscle relaxation, using grounding techniques (such as identifying objects or sensations around you), taking short walks in nature, listening to calming music, and applying peppermint or lavender essential oils. Staying well-hydrated and maintaining a regular sleep schedule can also help manage headaches.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [25]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [26]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help alleviate lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees. Alternate between arching your back up (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while engaging your core. Hold each extension for 5 seconds. Complete 10 repetitions on each side.\n\n- **Partial Crunches:** Lie on your back with knees bent, arms crossed over your chest, tighten your stomach muscles, and raise your shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat on the floor. Hold for 15-30 seconds, then switch legs.\n\n- **Pelvic Tilts:** Lie on your back with knees bent. Flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThese exercises are gentle and ai

In [28]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. According to the provided information, sleep is essential for physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Additionally, adults typically need 7-9 hours of quality sleep per night to support these functions. Poor sleep or sleep disturbances, such as insomnia, can negatively affect physical health, mental state, and immune function. Therefore, maintaining good sleep hygiene and creating an optimal sleep environment are important strategies to promote overall health.'

In [29]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water and staying hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using essential oils like peppermint or lavender\n- Engaging in relaxation techniques such as deep breathing, progressive muscle relaxation, or grounding exercises\n- Taking short walks, preferably in nature\n- Listening to calming music\n\nAdditionally, maintaining a regular sleep schedule, practicing mindfulness and meditation, and managing stress through healthy lifestyle habits can help alleviate headaches and reduce stress overall.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

It would help improve recall as multiple varied queires will likely increase the search space -- thereby increasing the likelyhood of finding more relvenat response and hence better recall. 

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [30]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [31]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [32]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [33]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [34]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include gentle stretches and strengthening movements such as:\n\n- Cat-Cow Stretch: On hands and knees, alternate arching your back up (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Partial Crunches: Lie on your back with knees bent, arms crossed over your chest, and tighten your stomach muscles to lift shoulders off the floor. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest, hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThese exercises can help alleviate discomfort and prevent future episodes of lower back pain.'

In [36]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly impacts overall health by supporting physical repair, mental well-being, and cognitive functions. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep—typically 7 to 9 hours per night for adults—ensures these restorative processes occur effectively. Poor sleep quality or insufficient sleep can lead to fatigue, low energy, headaches, and increased susceptibility to illness, affecting both physical health and mental clarity. Practicing good sleep hygiene, such as maintaining a consistent sleep schedule, creating a relaxing bedtime routine, and optimizing the sleep environment, can improve sleep quality and thereby enhance overall health.'

In [37]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing relaxation techniques such as deep breathing and progressive muscle relaxation, engaging in mindfulness or meditation, taking gentle walks in nature, listening to calming music, and applying essential oils like peppermint or lavender. Additionally, maintaining good hydration by drinking water, getting adequate sleep through a consistent routine, and managing stress through hobbies and social support can help alleviate stress-related headaches.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [38]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [39]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [40]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\nThese gentle stretching and strengthening exercises are recommended to alleviate discomfort and help prevent future episode

In [41]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly impacts overall health by supporting physical, mental, and cognitive well-being. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep—typically 7 to 9 hours per night—helps strengthen the immune system, improve mood, enhance concentration, and reduce the risk of chronic conditions such as headaches, digestive issues, and stress-related illnesses. Maintaining good sleep hygiene and creating a restful sleep environment are essential practices to ensure quality sleep and promote overall health.'

In [42]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises such as inhaling for 4 counts, holding for 4, and exhaling for 4.\n- Progressive muscle relaxation, which involves tensing and releasing muscle groups from toes to head.\n- Grounding techniques, like naming 5 things you see, 4 you hear, 3 you feel, 2 you smell, and 1 you taste.\n- Taking short walks, preferably in nature.\n- Listening to calming music.\n- Applying cold or warm compresses to the head or neck.\n- Resting in a dark, quiet room.\n- Gentle massage of temples and neck.\n- Using essential oils such as peppermint or lavender.\n- Staying well-hydrated by drinking water.\n- Maintaining a regular sleep schedule.\n\nThese approaches can help alleviate stress and headaches naturally.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [43]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [44]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [45]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [46]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [47]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [48]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting the pelvis slightly. Hold for 10 seconds, repeat 8-12 times.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. \n\nThese exercises, when performed gently and regularly, can help alleviate lower back discomfort and prevent future issues.'

In [49]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly impacts overall health in multiple ways. During sleep, your body repairs tissues, which supports physical health and recovery. Sleep is also essential for mental well-being and cognitive functions, such as memory consolidation and learning. Additionally, sleep regulates hormones related to growth and appetite, helping maintain a healthy weight and metabolic balance. Generally, adults need 7-9 hours of quality sleep per night to support these vital processes and promote overall wellness. Poor sleep or insufficient rest can lead to issues like fatigue, headaches, weakened immune function, and increased risk for various chronic conditions.'

In [50]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises: Inhale for 4 counts, hold for 4, exhale for 4, and hold for 4 to promote relaxation.\n- Progressive muscle relaxation: Tense and release muscle groups from toes to head to reduce tension.\n- Grounding techniques: Name five things you see, four you hear, three you feel, two you smell, and one you taste to stay present.\n- Rest and sleep hygiene practices: Creating a relaxing bedtime routine, keeping your sleep environment cool, dark, and quiet, and maintaining a consistent sleep schedule.\n- Herbal teas such as chamomile or valerian root which can promote relaxation.\n- Applying cold or warm compresses to the head or neck.\n- Gentle massage of temples and neck muscles.\n- Using essential oils like peppermint or lavender for their calming effects.\n- Engaging in mindfulness or meditation practices to reduce stress and improve emotional well-being.\n\nIf headaches are triggered by dehydration, stress, 

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

This scenario would result in lots of similar FAQs (say around refind policy, refind timelines, refund eligible) will get bundled in same chunk thereby leading the precisions issues. Also tken cost will be higher for e.g., given any refund question all like FAQs will be pulled in context. To handle this situation, there are various mechansim we can employ - like setting a context size (if hits that, store in separate chunk), setting a limit on percentile (e.g. < .85 score, store separately etc) 

---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [51]:
from dotenv import load_dotenv
import os

# Load environment variables from .env
load_dotenv()

# Enable tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"

assert os.getenv("LANGCHAIN_API_KEY"), "LANGCHAIN_API_KEY not found!"

In [52]:
os.environ["LANGCHAIN_PROJECT"] = "advanced-retriever-eval"

In [53]:
### YOUR CODE HERE
import pandas as pd
from tqdm import tqdm

from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.dataset_schema import EvaluationDataset

from ragas.metrics import (
    LLMContextRecall,
    Faithfulness,
    FactualCorrectness,
    ResponseRelevancy,
    ContextEntityRecall,
    NoiseSensitivity
)


from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI


/Users/rahul/Documents/AIE9/11_Advanced_Retrieval/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/Users/rahul/Documents/AIE9/11_Advanced_Retrieval/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/Users/rahul/Documents/AIE9/11_Advanced_Retrieval/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


In [54]:
import ragas
print(ragas.__version__)
print(ragas.__file__)

0.2.10
/Users/rahul/Documents/AIE9/11_Advanced_Retrieval/.venv/lib/python3.13/site-packages/ragas/__init__.py


In [59]:
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [69]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

eval_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=100,
)

eval_docs = eval_splitter.split_documents(raw_docs)

In [74]:

from ragas.testset import TestsetGenerator
from langchain_openai import ChatOpenAI, OpenAIEmbeddings


generator_llm = ChatOpenAI(model="gpt-4.1-mini")

generator = TestsetGenerator.from_langchain(
    llm=generator_llm,
    embedding_model=OpenAIEmbeddings(model="text-embedding-3-small"),
)

testset = generator.generate_with_langchain_docs(
    eval_docs,
    testset_size=5,
)

from ragas.dataset_schema import EvaluationDataset

dataset = EvaluationDataset.from_pandas(testset.to_pandas())

Applying SummaryExtractor:   0%|          | 0/16 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/16 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/48 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/6 [00:00<?, ?it/s]

In [75]:
dataset.to_pandas().head()

,user_input,reference_contexts,reference
0,How does incorporating flexibility exercises i...,[The Personal Wellness Guide\nA Comprehensive ...,Flexibility exercises are one of the four main...
1,How do I perform Pelvic Tilts correctly to hel...,[Recommended exercises for lower back pain inc...,"To perform Pelvic Tilts, lie on your back with..."
2,What exercises are recommended for Monday in a...,[Neck and Shoulder Tension\nDesk work and poor...,"On Monday, the beginner weekly schedule recomm..."
3,how chapter 13 explain the habit loop help bui...,[<1-hop>\n\nSample wellness morning routine:\n...,Chapter 13 explains that habits form through a...
4,Hw can I use Chapter 15 evening wind-down rout...,[<1-hop>\n\nSample wellness morning routine:\n...,Chapter 15 recommends an effective evening win...


In [76]:
evaluator_llm = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-4.1-mini")
)

custom_run_config = RunConfig(timeout=360)

In [84]:
def build_evaluation_dataset(retriever, dataset):
    rows = []

    df = dataset.to_pandas()

    for row in tqdm(df.itertuples(), total=len(df)):
        question = row.user_input
        ground_truth = row.reference

        # Retrieve documents
        docs = retriever.invoke(question)
        retrieved_contexts = [doc.page_content for doc in docs]

        # Generate answer
        response = (rag_prompt | chat_model).invoke(
            {"context": retrieved_contexts, "question": question}
        )

        rows.append({
            "user_input": question,
            "response": response.content,
            "retrieved_contexts": retrieved_contexts,
            "reference": ground_truth,
        })

    return EvaluationDataset.from_pandas(pd.DataFrame(rows))

In [85]:
def evaluate_retriever(retriever, dataset, name):
    print(f"\n🔍 Evaluating Retriever: {name}")

    evaluation_dataset = build_evaluation_dataset(retriever, dataset)

    result = evaluate(
        dataset=evaluation_dataset,
        metrics=[
            LLMContextRecall(),
            Faithfulness(),
            FactualCorrectness(),
            ResponseRelevancy(),
            ContextEntityRecall(),
            NoiseSensitivity()
        ],
        llm=evaluator_llm,
        run_config=custom_run_config
    )

    print(result)

    return result

In [86]:
retrievers = {
    "BM25": bm25_retriever,
    "Naive": naive_retriever,
    "Semantic": semantic_retriever,
    "ParentDoc": parent_document_retriever,
    "Compression": compression_retriever,
    "MultiQuery": multi_query_retriever,
    "Ensemble": ensemble_retriever,
}

In [87]:
dataset.to_pandas().columns

Index(['user_input', 'reference_contexts', 'reference'], dtype='str')

In [88]:
all_results = {}

for name, retriever in retrievers.items():
    result = evaluate_retriever(retriever, dataset, name)
    all_results[name] = result


🔍 Evaluating Retriever: BM25


100%|██████████| 6/6 [00:08<00:00,  1.46s/it]


Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

{'context_recall': 0.4167, 'faithfulness': 0.3439, 'factual_correctness': 0.3817, 'answer_relevancy': 0.8106, 'context_entity_recall': 0.2015, 'noise_sensitivity_relevant': 0.0677}

🔍 Evaluating Retriever: Naive


100%|██████████| 6/6 [00:12<00:00,  2.10s/it]


...

{'context_recall': 0.8750, 'faithfulness': 0.4947, 'factual_correctness': 0.4850, 'answer_relevancy': 0.9774, 'context_entity_recall': 0.3784, 'noise_sensitivity_relevant': 0.1330}

🔍 Evaluating Retriever: Ensemble


100%|██████████| 6/6 [00:26<00:00,  4.45s/it]


Evaluating:   0%|          | 0/36 [00:00<?, ?it/s]

Exception raised in Job[5]: TimeoutError()
Exception raised in Job[23]: TimeoutError()
Exception raised in Job[29]: TimeoutError()
Exception raised in Job[35]: TimeoutError()


{'context_recall': 0.9167, 'faithfulness': 0.5600, 'factual_correctness': 0.5583, 'answer_relevancy': 0.9694, 'context_entity_recall': 0.3550, 'noise_sensitivity_relevant': 0.0625}


| Retriever       | Context Recall | Faithfulness | Factual Correctness | Answer Relevancy | Context Entity Recall | Noise Sensitivity | Latency (s) | Tokens Used | Cost ($)   |
| --------------- | -------------- | ------------ | ------------------- | ---------------- | --------------------- | ----------------- | ----------- | ----------- | ---------- |
| **BM25**        | 0.4167         | 0.3439       | 0.3817              | 0.8106           | 0.2015                | 0.0677       | 190.45      | 222,449     | 0.1898     |
| **Naive**       | 0.8750         | **0.6084**   | 0.6033              | 0.9759           | 0.3163                | 0.0834            | 376.38      | 381,121     | 0.3350     |
| **Semantic**    | 0.8611         | 0.4071       | 0.4617              | 0.9719           | 0.1933                | 0.0944            | 292.96      | 414,789     | 0.3135     |
| **ParentDoc**   | 0.8194         | 0.5266       | **0.6067**          | 0.8175           | 0.2579                | 0.1222            | 184.24      | 208,487     | 0.1550     |
| **Compression** | 0.6250         | 0.3297       | 0.5150              | 0.9757           | **0.4075**            | 0.0618            | **158.46**  | **198,292** | **0.1538** |
| **MultiQuery**  | 0.8750         | 0.4947       | 0.4850              | **0.9774**       | 0.3784                | 0.1330            | 384.33      | 426,661     | 0.3589     |
| **Ensemble**    | **0.9167**     | 0.5600       | 0.5583              | 0.9694           | 0.3550                | **0.0625**        | 388.15      | 454,665     | 0.3639     |

**Retriever Evaluation Interpretation**

Across all six RAGAS metrics, the Ensemble retriever achieved the highest context recall (0.9167) while maintaining strong faithfulness (0.5600) and factual correctness (0.5583). It also demonstrated low noise sensitivity, indicating robust retrieval. However, this came at the highest computational cost ($0.3639) and latency (388s), making it the most expensive strategy.

The ParentDoc retriever emerged as the most balanced option. It achieved the highest factual correctness (0.6067), strong faithfulness (0.5266), and competitive recall (0.8194) while maintaining significantly lower cost ($0.1550) and latency (184s). This makes it the strongest production candidate for this dataset.

Compression retrieval was the most efficient approach, with the lowest latency and cost and the highest entity-level recall, though it sacrificed overall recall and faithfulness.

Given the structured and concept-heavy nature of the wellness corpus, larger contextual windows (ParentDoc) provide strong grounding and factual consistency without the cost overhead of ensemble or multi-query approaches.

Therefore, while Ensemble maximizes retrieval quality, ParentDoc provides the best overall tradeoff between performance, cost, and latency for practical deployment and hence is best suited for Wellness scenario.